In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Load datasets
df = pd.read_csv('data/Billionaires Statistics Dataset.csv')       # 2023 snapshot
df_hist = pd.read_csv('data/billionaires.csv')                     # historical 1996-2014

print(f'2023 snapshot : {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Historical    : {df_hist.shape[0]} rows, {df_hist.shape[1]} columns')

2023 snapshot : 2640 rows, 35 columns
Historical    : 2614 rows, 22 columns


Where do billionaire live (chloropeth map)

In [ ]:
# All billionaires by country - Choropleth Map
import plotly.graph_objects as go
import plotly.express as px

# Get all billionaires
all_billionaires = df.copy()

# Group by country and aggregate
billionaires_by_country = all_billionaires.groupby('country').agg({
    'finalWorth': 'sum',
    'personName': 'count'
}).reset_index()

# Rename columns for clarity
billionaires_by_country.columns = ['Country', 'Total_Worth', 'Count']

# Convert finalWorth to numeric (remove any non-numeric characters if present)
billionaires_by_country['Total_Worth'] = pd.to_numeric(billionaires_by_country['Total_Worth'], errors='coerce')




All Billionaires Distribution by Country:
           Country  Total_Worth  Count
74   United States      4575100    754
16           China      1805500    523
31           India       628700    157
24          France       499500     35
26         Germany       462100    102
65     Switzerland       409900     78
73  United Kingdom       370700     82
58          Russia       351000     79
29       Hong Kong       321500     68
13          Canada       173900     42

Total countries represented: 78


In [8]:
# Create Choropleth Map with Emerald Green Color
# First, map country names to ISO-3 codes
import pycountry

country_iso = {}
for idx, country in enumerate(billionaires_by_country['Country']):
    try:
        country_iso[country] = pycountry.countries.search_fuzzy(country)[0].alpha_3
    except:
        # Handle special cases
        special_cases = {
            'United States': 'USA',
            'United Kingdom': 'GBR',
            'China': 'CHN',
            'Hong Kong': 'HKG',
            'Taiwan': 'TWN',
            'Russia': 'RUS',
            'South Korea': 'KOR'
        }
        country_iso[country] = special_cases.get(country, country)

billionaires_by_country['ISO_Code'] = billionaires_by_country['Country'].map(country_iso)

# Emerald green palette
emerald_colorscale = [
    [0, '#FFFFFF'],           # White for low values
    [0.3, '#A8E6CF'],         # Light emerald
    [0.6, '#56D8A0'],         # Medium emerald
    [1, '#1B5E4A']            # Dark emerald for highest values
]

fig = go.Figure(data=go.Choropleth(
    locations=billionaires_by_country['ISO_Code'],
    z=billionaires_by_country['Total_Worth'],
    text=billionaires_by_country['Country'],
    customdata=billionaires_by_country[['Count', 'Total_Worth']],
    hovertemplate='<b>%{text}</b><br>' +
                  'Number of Billionaires: %{customdata[0]:.0f}<br>' +
                  'Total Wealth (USD Billions): %{customdata[1]:,.0f}<br>' +
                  '<extra></extra>',
    colorscale=emerald_colorscale,
    colorbar=dict(
        title="Total Wealth<br>(USD Billions)",
        thickness=15,
        len=0.7
    )
))

fig.update_layout(
    title_text='<b>All Billionaires Distribution by Country</b><br><sub>Wealth Aggregated by Country (Emerald Green = Greater Wealth)</sub>',
    geo=dict(
        projection_type='natural earth',
        showland=True,
        landcolor='rgb(243, 243, 243)',
        coastlinecolor='rgb(204, 204, 204)',
        showocean=True,
        oceancolor='rgb(100, 180, 220)'
    ),
    height=700,
    font=dict(size=11)
)

fig.show()

## Are Billionaire born on same day?

In [13]:
# Visualization - Top 25 Most Common Billionaire Birthdays
import plotly.graph_objects as go

# Get top 25 birthdays
top_birthdays = birthday_counts.head(25)

# Emerald green color gradient
emerald_colors = ['#1B5E4A' if i == 0 else '#56D8A0' if i < 12 else '#A8E6CF' 
                  for i in range(len(top_birthdays))]

fig = go.Figure(data=[
    go.Bar(
        y=top_birthdays['month_day'],
        x=top_birthdays['Count'],
        orientation='h',
        marker=dict(
            color=top_birthdays['Count'],
            colorscale=[
                [0, '#A8E6CF'],        # Light emerald
                [0.5, '#56D8A0'],      # Medium emerald
                [1, '#1B5E4A']         # Dark emerald
            ],
            showscale=True,
            colorbar=dict(
                title="Number of<br>Billionaires",
                thickness=15,
                len=0.7
            )
        ),
        hovertemplate='<b>%{y}</b><br>' +
                      'Billionaires: %{x}<br>' +
                      '<extra></extra>',
        text=top_birthdays['Count'],
        textposition='outside'
    )
])

fig.update_layout(
    title_text='<b>Top 25 Most Common Billionaire Birthdays</b><br><sub>Emerald Green Intensity = More Billionaires Share This Birthday</sub>',
    xaxis_title='Number of Billionaires',
    yaxis_title='Birthday (Month Day)',
    hovermode='y unified',
    height=700,
    width=900,
    template='plotly_white',
    font=dict(size=11),
    showlegend=False
)

fig.show()


## Top Billionaires Leaderboard

In [18]:
# Top 10 Billionaires Profile Cards - Visual Leaderboard with Plotly
import plotly.graph_objects as go

# Prepare data for visual cards
df_top_10 = df.nlargest(10, 'finalWorth')[['rank', 'personName', 'finalWorth', 'source', 'country']].copy()
df_top_10['finalWorth'] = pd.to_numeric(df_top_10['finalWorth'], errors='coerce')
df_top_10['finalWorth'] = df_top_10['finalWorth'].round(0).astype(int)
df_top_10 = df_top_10.reset_index(drop=True)

# Create a more visual representation using scatter plot styled as cards
fig = go.Figure()

# Add bars showing wealth
fig.add_trace(go.Bar(
    y=df_top_10['personName'],
    x=df_top_10['finalWorth'],
    orientation='h',
    marker=dict(
        color=df_top_10['finalWorth'],
        colorscale=[
            [0, '#A8E6CF'],        # Light emerald
            [0.5, '#56D8A0'],      # Medium emerald
            [1, '#1B5E4A']         # Dark emerald
        ],
        showscale=True,
        colorbar=dict(
            title="Wealth<br>(USD B)",
            thickness=15,
            len=0.7
        ),
        line=dict(color='#1B5E4A', width=2)
    ),
    text=[f"<b>#{int(rank)}</b><br>${val}B<br>{src}" 
          for rank, val, src in zip(df_top_10['rank'], df_top_10['finalWorth'], df_top_10['source'])],
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>' +
                  'Rank: #%{customdata[0]}<br>' +
                  'Wealth: $%{x}B<br>' +
                  'Source: %{customdata[1]}<br>' +
                  'Country: %{customdata[2]}<br>' +
                  '<extra></extra>',
    customdata=df_top_10[['rank', 'source', 'country']].values
))

fig.update_layout(
    title_text='<b>Top 10 Billionaires Profile Cards</b><br><sub>Ranked by Total Wealth - Emerald Green Scale</sub>',
    xaxis_title='Total Wealth (USD Billions)',
    yaxis_title='Billionaire Name',
    height=700,
    width=1100,
    hovermode='closest',
    template='plotly_white',
    font=dict(size=12, color='#1B5E4A', family='Arial'),
    paper_bgcolor='#F0F8F5',
    plot_bgcolor='#FFFFFF',
    margin=dict(l=200, r=100, t=150, b=60),
    yaxis=dict(tickfont=dict(size=11, color='#1B5E4A', family='Arial')),
    xaxis=dict(tickfont=dict(size=10, color='#1B5E4A'))
)

fig.update_yaxes(autorange="reversed")

fig.show()



## Age Analysis of Billionaires

In [23]:
# Age Analysis - Data Preparation
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Prepare age data
df_age = df[['personName', 'age', 'finalWorth', 'country', 'source']].copy()
df_age['finalWorth'] = pd.to_numeric(df_age['finalWorth'], errors='coerce')
df_age = df_age.dropna(subset=['age', 'finalWorth'])

# Convert age to numeric
df_age['age'] = pd.to_numeric(df_age['age'], errors='coerce')
df_age = df_age.dropna(subset=['age'])

print("\n" + "="*80)
print("BILLIONAIRE AGE ANALYSIS - STATISTICS")
print("="*80)
print(f"\nTotal billionaires with age data: {len(df_age)}")
print(f"\nAge Statistics:")
print(f"  Minimum Age: {df_age['age'].min():.0f} years")
print(f"  Maximum Age: {df_age['age'].max():.0f} years")
print(f"  Mean Age: {df_age['age'].mean():.1f} years")
print(f"  Median Age: {df_age['age'].median():.1f} years")
print(f"  Mode Age: {df_age['age'].mode().values[0]:.0f} years")
print(f"  Std Dev: {df_age['age'].std():.1f} years")
print(f"\nWealth vs Age:")
print(f"  Average Wealth: ${df_age['finalWorth'].mean():,.0f}B")
print(f"  Correlation (Age vs Wealth): {df_age['age'].corr(df_age['finalWorth']):.3f}")

# Age groups analysis
df_age['age_group'] = pd.cut(df_age['age'], 
                             bins=[0, 30, 40, 50, 60, 70, 80, 100],
                             labels=['<30', '30-40', '40-50', '50-60', '60-70', '70-80', '80+'])

age_group_stats = df_age.groupby('age_group', observed=True).agg({
    'personName': 'count',
    'finalWorth': ['sum', 'mean', 'median']
}).round(0)

print(f"\nAge Group Distribution:")
print("-" * 80)
age_group_stats.columns = ['Count', 'Total_Wealth', 'Avg_Wealth', 'Median_Wealth']
print(age_group_stats.to_string())



BILLIONAIRE AGE ANALYSIS - STATISTICS

Total billionaires with age data: 2575

Age Statistics:
  Minimum Age: 18 years
  Maximum Age: 101 years
  Mean Age: 65.1 years
  Median Age: 65.0 years
  Mode Age: 60 years
  Std Dev: 13.3 years

Wealth vs Age:
  Average Wealth: $4,679B
  Correlation (Age vs Wealth): 0.067

Age Group Distribution:
--------------------------------------------------------------------------------
           Count  Total_Wealth  Avg_Wealth  Median_Wealth
age_group                                                
<30           15         64000     4267.00        1700.00
30-40         70        315300     4504.00        2000.00
40-50        239        865200     3620.00        2000.00
50-60        679       2918400     4298.00        2200.00
60-70        660       2859600     4333.00        2400.00
70-80        573       2860000     4991.00        2600.00
80+          338       2165300     6406.00        3000.00


In [28]:
# Visualization 5: Violin Plot - Age Distribution by Age Group
fig_violin = go.Figure()

for i, age_group in enumerate(['<30', '30-40', '40-50', '50-60', '60-70', '70-80', '80+']):
    data = df_age[df_age['age_group'] == age_group]['finalWorth']
    
    if len(data) > 0:
        fig_violin.add_trace(go.Violin(
            x=[age_group] * len(data),
            y=data,
            name=age_group,
            box_visible=True,
            meanline_visible=True,
            fillcolor=['#A8E6CF', '#B5EBD8', '#56D8A0', '#4ACCA0', '#1B5E4A', '#0D3D2E', '#043D2C'][i],
            line_color='#1B5E4A',
            opacity=0.8,
            hovertemplate='Age Group: %{x}<br>Wealth: $%{y:.0f}B<extra></extra>'
        ))

fig_violin.update_layout(
    title_text='<b>Wealth Distribution by Billionaire Age Group</b><br><sub>Violin Plot with Box & Mean Line</sub>',
    xaxis_title='Age Group',
    yaxis_title='Total Wealth (USD Billions)',
    height=600,
    width=1100,
    template='plotly_white',
    font=dict(size=11, color='#1B5E4A', family='Arial'),
    paper_bgcolor='#F0F8F5',
    plot_bgcolor='#FFFFFF',
    showlegend=False,
    hovermode='closest'
)

fig_violin.show()

print("\n" + "="*80)
print("AGE GROUP WEALTH ANALYSIS")
print("="*80)
print(df_age_group_summary.to_string(index=False))
print("="*80)



AGE GROUP WEALTH ANALYSIS
Age_Group  Count  Avg_Wealth  Avg_Age
      <30     15     4266.67    25.47
    30-40     70     4504.29    37.33
    40-50    239     3620.08    46.33
    50-60    679     4298.09    56.06
    60-70    660     4332.73    65.58
    70-80    573     4991.27    75.16
      80+    338     6406.21    86.25


## GDP vs Billionaire Count Analysis
Validate if richer countries produce more billionaires

In [30]:
# Data Preparation: GDP vs Billionaire Count
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Get billionaires by country
billionaires_count = df.groupby('country').agg({
    'personName': 'count',
    'finalWorth': 'sum'
}).reset_index()
billionaires_count.columns = ['Country', 'Billionaire_Count', 'Total_Wealth']
billionaires_count['Total_Wealth'] = pd.to_numeric(billionaires_count['Total_Wealth'], errors='coerce')

# GDP data (2023 estimates in USD Billions)
# Source: IMF World Economic Outlook
gdp_data = {
    'United States': 27360, 'China': 17734, 'Japan': 4231, 'Germany': 4080,
    'India': 3736, 'United Kingdom': 3332, 'France': 2783, 'Italy': 2010,
    'Brazil': 1920, 'Canada': 2117, 'South Korea': 1642, 'Spain': 1390,
    'Australia': 1687, 'Mexico': 1313, 'Netherlands': 1257, 'Saudi Arabia': 1108,
    'Switzerland': 957, 'Turkey': 1163, 'Poland': 688, 'Belgium': 688,
    'Sweden': 615, 'Argentina': 632, 'Austria': 516, 'Norway': 598,
    'Nigeria': 477, 'Iran': 374, 'Israel': 529, 'Ireland': 530,
    'United Arab Emirates': 509, 'Singapore': 592, 'Hong Kong': 397,
    'Thailand': 504, 'Malaysia': 438, 'Indonesia': 1119, 'Philippines': 548,
    'Vietnam': 429, 'Pakistan': 376, 'Bangladesh': 416, 'Egypt': 437,
    'Russia': 1860, 'Ukraine': 184, 'Czech Republic': 296, 'Portugal': 267,
    'Greece': 219, 'Chile': 314, 'Colombia': 331, 'Peru': 245,
    'New Zealand': 249, 'Denmark': 405, 'Finland': 298, 'Romania': 301,
    'Taiwan': 764, 'Hong Kong': 397, 'Kazakhstan': 239, 'Kuwait': 186,
    'Qatar': 240, 'Bahrain': 43, 'Oman': 120, 'Malta': 19,
    'Luxembourg': 84, 'Cyprus': 32, 'Sri Lanka': 84, 'Cyprus': 32,
    'Jordan': 45, 'Lebanon': 22
}

# Merge GDP with billionaire data
gdp_df = pd.DataFrame(list(gdp_data.items()), columns=['Country', 'GDP'])
analysis_df = billionaires_count.merge(gdp_df, on='Country', how='inner')

print("\n" + "="*100)
print("GDP vs BILLIONAIRE COUNT ANALYSIS")
print("="*100)
print(f"\nCountries analyzed: {len(analysis_df)}")
print(f"\nCorrelation between GDP and Billionaire Count: {analysis_df['GDP'].corr(analysis_df['Billionaire_Count']):.3f}")
print(f"Correlation between GDP and Total Wealth: {analysis_df['GDP'].corr(analysis_df['Total_Wealth']):.3f}")

print("\n" + "-"*100)
print("Top 15 Countries by GDP and Billionaire Count:")
print("-"*100)
top_analysis = analysis_df.nlargest(15, 'GDP')[['Country', 'GDP', 'Billionaire_Count', 'Total_Wealth']]
print(top_analysis.to_string(index=False))
print("="*100)



GDP vs BILLIONAIRE COUNT ANALYSIS

Countries analyzed: 55

Correlation between GDP and Billionaire Count: 0.985
Correlation between GDP and Total Wealth: 0.968

----------------------------------------------------------------------------------------------------
Top 15 Countries by GDP and Billionaire Count:
----------------------------------------------------------------------------------------------------
       Country   GDP  Billionaire_Count  Total_Wealth
 United States 27360                754       4575100
         China 17734                523       1805500
         Japan  4231                 38        146800
       Germany  4080                102        462100
         India  3736                157        628700
United Kingdom  3332                 82        370700
        France  2783                 35        499500
        Canada  2117                 42        173900
         Italy  2010                 55        156900
        Brazil  1920                 44        10

In [33]:
# Visualization 3: Bubble Chart - GDP per Billionaire vs Billionaire Efficiency
analysis_df['GDP_per_Billionaire'] = analysis_df['GDP'] / analysis_df['Billionaire_Count']

# Sort by GDP per billionaire to find countries that produce billionaires efficiently
efficiency_ranking = analysis_df[['Country', 'GDP', 'Billionaire_Count', 'GDP_per_Billionaire', 'Total_Wealth']].sort_values('GDP_per_Billionaire')

fig_bubble = go.Figure()

fig_bubble.add_trace(go.Scatter(
    x=analysis_df['GDP'],
    y=analysis_df['Billionaire_Count'],
    mode='markers',
    text=analysis_df['Country'],
    marker=dict(
        size=analysis_df['Billionaire_Count'] * 0.8,  # Size by billionaire count
        color=analysis_df['GDP_per_Billionaire'],
        colorscale=[
            [0, '#1B5E4A'],        # Dark emerald = efficient (low GDP per billionaire)
            [1, '#A8E6CF']         # Light emerald = less efficient (high GDP per billionaire)
        ],
        colorbar=dict(
            title="GDP per<br>Billionaire<br>(Billions USD)",
            thickness=15,
            len=0.7
        ),
        line=dict(color='#1B5E4A', width=1),
        showscale=True
    ),
    hovertemplate='<b>%{text}</b><br>' +
                  'GDP: $%{x:.0f}B<br>' +
                  'Billionaires: %{y}<br>' +
                  'GDP per Billionaire: $%{customdata[0]:.1f}B<br>' +
                  'Total Wealth: $%{customdata[1]:.0f}B<br>' +
                  '<extra></extra>',
    customdata=analysis_df[['GDP_per_Billionaire', 'Total_Wealth']]
))

fig_bubble.update_layout(
    title_text='<b>Billionaire Production Efficiency</b><br><sub>Dark Emerald = More Efficient at Creating Billionaires | Bubble Size = Billionaire Count</sub>',
    xaxis_title='Country GDP (USD Billions)',
    yaxis_title='Number of Billionaires',
    height=700,
    width=1200,
    template='plotly_white',
    font=dict(size=11, color='#1B5E4A'),
    paper_bgcolor='#F0F8F5',
    plot_bgcolor='#FFFFFF',
    hovermode='closest'
)

fig_bubble.show()

print("\n" + "="*100)
print("BILLIONAIRE PRODUCTION EFFICIENCY ANALYSIS")
print("="*100)
print("\nMost Efficient Countries (Lowest GDP per Billionaire):")
print("-"*100)
most_efficient = efficiency_ranking.head(10)
print(most_efficient.to_string(index=False))

print("\n\nLeast Efficient Countries (Highest GDP per Billionaire):")
print("-"*100)
least_efficient = efficiency_ranking.tail(10)
print(least_efficient.to_string(index=False))
print("="*100)

print("\n📊 KEY INSIGHTS:")
print("  • Countries with LOWER 'GDP per Billionaire' are MORE efficient at creating billionaires")
print("  • This shows which economies convert wealth into billionaire entrepreneurship most effectively")



BILLIONAIRE PRODUCTION EFFICIENCY ANALYSIS

Most Efficient Countries (Lowest GDP per Billionaire):
----------------------------------------------------------------------------------------------------
    Country  GDP  Billionaire_Count  GDP_per_Billionaire  Total_Wealth
  Hong Kong  397                 68                 5.84        321500
     Cyprus   32                  5                 6.40          9600
    Lebanon   22                  2                11.00          5600
Switzerland  957                 78                12.27        409900
  Singapore  592                 46                12.87        138000
     Taiwan  764                 43                17.77        117900
   Thailand  504                 28                18.00        100700
     Israel  529                 26                20.35         72500
     Russia 1860                 79                23.54        351000
     Sweden  615                 26                23.65         90900


Least Efficient 

## Education vs Billionaires Analysis
Does higher education enrollment correlate with billionaire creation?

In [34]:
# Data Preparation: Education vs Billionaire Count
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Get billionaires and education data by country
df_education = df[['country', 'personName', 'finalWorth', 'gross_tertiary_education_enrollment']].copy()
df_education['finalWorth'] = pd.to_numeric(df_education['finalWorth'], errors='coerce')
df_education['gross_tertiary_education_enrollment'] = pd.to_numeric(df_education['gross_tertiary_education_enrollment'], errors='coerce')

# Group by country
education_analysis = df_education.groupby('country').agg({
    'personName': 'count',
    'finalWorth': 'sum',
    'gross_tertiary_education_enrollment': 'first'  # Same for all billionaires in same country
}).reset_index()

education_analysis.columns = ['Country', 'Billionaire_Count', 'Total_Wealth', 'Education_Enrollment']

# Remove rows with missing education data
education_analysis = education_analysis.dropna(subset=['Education_Enrollment'])

print("\n" + "="*100)
print("EDUCATION vs BILLIONAIRE COUNT ANALYSIS")
print("="*100)
print(f"\nCountries analyzed: {len(education_analysis)}")
print(f"\nEducation Enrollment Statistics:")
print(f"  Min: {education_analysis['Education_Enrollment'].min():.1f}%")
print(f"  Max: {education_analysis['Education_Enrollment'].max():.1f}%")
print(f"  Mean: {education_analysis['Education_Enrollment'].mean():.1f}%")
print(f"  Median: {education_analysis['Education_Enrollment'].median():.1f}%")

print(f"\nCorrelation between Education Enrollment and Billionaire Count: {education_analysis['Education_Enrollment'].corr(education_analysis['Billionaire_Count']):.3f}")
print(f"Correlation between Education Enrollment and Total Wealth: {education_analysis['Education_Enrollment'].corr(education_analysis['Total_Wealth']):.3f}")

print("\n" + "-"*100)
print("Top 15 Countries by Education Enrollment and Billionaire Count:")
print("-"*100)
top_education = education_analysis.nlargest(15, 'Education_Enrollment')[['Country', 'Education_Enrollment', 'Billionaire_Count', 'Total_Wealth']]
print(top_education.to_string(index=False))
print("="*100)



EDUCATION vs BILLIONAIRE COUNT ANALYSIS

Countries analyzed: 66

Education Enrollment Statistics:
  Min: 4.0%
  Max: 136.6%
  Mean: 57.5%
  Median: 60.9%

Correlation between Education Enrollment and Billionaire Count: 0.122
Correlation between Education Enrollment and Total Wealth: 0.140

----------------------------------------------------------------------------------------------------
Top 15 Countries by Education Enrollment and Billionaire Count:
----------------------------------------------------------------------------------------------------
      Country  Education_Enrollment  Billionaire_Count  Total_Wealth
       Greece                136.60                  3          9100
    Australia                113.10                 43        173500
  South Korea                 94.30                 29         81400
    Argentina                 90.00                  4         11000
        Spain                 88.90                 25        133700
        Chile               